# Week 14: LLM Fine-Tuning & Quantization
## Notebook 10: FP8 Mixed-Precision Training: E4M3 vs E5M2 Formats

### 1. Conceptual Intuition & Theoretical Background
While FP16 and BF16 use 16 bits per parameter, **8-bit Floating Point (FP8)** halves memory footprint and doubles tensor core throughput on NVIDIA Hopper (H100) and Blackwell GPUs.

**The Two Standard FP8 Formats (OCP Spec):**
1. **E4M3 (1 Sign, 4 Exponent, 3 Mantissa):**
   - Higher precision, smaller dynamic range (Max value: $\approx 448$).
   - Used for **Forward Pass Activations and Weights** where precision is critical.
2. **E5M2 (1 Sign, 5 Exponent, 2 Mantissa):**
   - Lower precision, wider dynamic range (Max value: $\approx 57,344$, identical dynamic range to FP16).
   - Used for **Backward Pass Gradients** where dynamic range prevents underflow.

**Delayed Dynamic Scaling:**
Because FP8 has limited dynamic range, tensors are scaled dynamically using scaling factors $S$:
$$x_{\text{fp8}} = \text{clip}\left( \text{round}(x \cdot S),\ -\text{Max},\ \text{Max} \right)$$

---

### 2. Scratch Python Implementation: FP8 Simulation & Scaling Engine

In [ ]:
import numpy as np

class FP8E4M3Simulator:
    """Simulates 8-bit floating point format with 4 exponent and 3 mantissa bits."""
    def __init__(self):
        self.max_val = 448.0
        self.min_val = -448.0
        # E4M3 has 3 mantissa bits -> 2^3 = 8 discrete levels between powers of 2
        self.quant_levels = 8.0
        
    def quantize(self, tensor, scale):
        scaled = tensor * scale
        clipped = np.clip(scaled, self.min_val, self.max_val)
        
        # Simulate quantization error
        # Exponent bucketing
        abs_val = np.maximum(np.abs(clipped), 1e-4)
        exponent = np.floor(np.log2(abs_val))
        step = (2.0 ** exponent) / self.quant_levels
        
        quantized = np.round(clipped / step) * step
        # Dequantize back to floating scale
        return quantized / scale

# Benchmark FP8 Quantization on Normal Distribution
np.random.seed(42)
weights_fp32 = np.random.normal(0, 0.05, 1000)

fp8_sim = FP8E4M3Simulator()
# Calculate optimal scaling factor S = Max_FP8 / amax(W)
amax = np.max(np.abs(weights_fp32))
optimal_scale = 448.0 / (amax + 1e-6)

weights_fp8_simulated = fp8_sim.quantize(weights_fp32, optimal_scale)
sq_error = np.mean((weights_fp32 - weights_fp8_simulated) ** 2)
snr = 10 * np.log10(np.var(weights_fp32) / (sq_error + 1e-12))

print("--- FP8 (E4M3) Simulation & Quantization ---")
print(f"Original FP32 Weights Range: [{weights_fp32.min():.4f}, {weights_fp32.max():.4f}]")
print(f"Optimal Delayed Scale S:     {optimal_scale:.2f}")
print(f"Reconstruction MSE:          {sq_error:.2e}")
print(f"Signal-to-Noise Ratio (SNR): {snr:.1f} dB (High fidelity preservation!)")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Range Scaling:** Scales small weights to fill the dynamic range of the format.
2. **Exponent Bucketing:** Simulates mantissa truncation across exponential intervals.
3. **Format Allocation:** Uses E4M3 for forward weights and E5M2 for backward gradients.

---

### 4. Industry Applications & Trade-offs
- **Modern Large-Scale Training (DeepSeek-V3, LLaMA-3.1):** Trained using FP8 mixed-precision to halve compute cluster costs.
- **Trade-off:** Poorly calibrated scaling factors cause numerical saturation or catastrophic gradient underflow.

---

### 5. Top Interview Questions & Answers
**Q1: Why is E4M3 preferred for forward pass weights while E5M2 is preferred for backward pass gradients?**
> **Answer:** Forward pass weights need higher precision (mantissa bits) to maintain model output quality. Backward pass gradients exhibit high variance and extreme spikes, requiring the wider dynamic range of E5M2 to prevent underflow and overflow.

**Q2: What is "Delayed Scaling" in FP8 training frameworks (like TransformerEngine)?**
> **Answer:** Computing exact scales for every tensor requires synchronizing GPU kernels. Delayed scaling uses the maximum absolute value from previous iterations to scale the current step, avoiding synchronization overhead.